# 🦆 Executing SQL on Tech News & Warehouse Data

You can run **pure SQL queries** directly on your Pandas DataFrames and CSV files using **DuckDB** or Python's built-in **SQLite** engine!

--- 
### **Method 1: DuckDB (Recommended for Warehouse Queries)**
DuckDB can query Pandas DataFrames and CSV files directly using standard ANSI SQL without copying data.

In [1]:
import duckdb
import pandas as pd

# Load our warehouse CSVs
dim_company = pd.read_csv('data/warehouse/dim_company.csv')
fct_article = pd.read_csv('data/warehouse/fct_article.csv')
fct_arr_observation = pd.read_csv('data/warehouse/fct_arr_observation.csv')

# -----------------------------------------------------
# Query 1: Join Fact Table with Dimension Table in SQL
# -----------------------------------------------------
query1 = """
SELECT 
    c.company_name,
    c.industry,
    a.observation_year,
    a.observation_quarter,
    a.arr_usd_M
FROM fct_arr_observation a
JOIN dim_company c ON a.company_id = c.company_id
WHERE c.company_name = 'Databricks'
ORDER BY a.observation_year, a.observation_quarter;
"""

df_result1 = duckdb.query(query1).df()
display(df_result1)

,company_name,industry,observation_year,observation_quarter,arr_usd_M
0,Databricks,Data Analytics,2020,2,420
1,Databricks,Data Analytics,2020,3,500
2,Databricks,Data Analytics,2021,1,740
3,Databricks,Data Analytics,2021,2,900
4,Databricks,Data Analytics,2021,4,1180
5,Databricks,Data Analytics,2022,3,1700
6,Databricks,Data Analytics,2023,2,2150
7,Databricks,Data Analytics,2023,2,2150
8,Databricks,Data Analytics,2023,4,2400
9,Databricks,Data Analytics,2023,4,2400


--- 
### **Query 2: GroupBy & Aggregations in SQL**
Find average revenue and total articles by industry.

In [2]:
query2 = """
SELECT 
    c.industry,
    COUNT(DISTINCT c.company_id) AS total_companies,
    COUNT(a.article_id) AS total_articles,
    ROUND(AVG(o.arr_usd_M), 1) AS avg_arr_usd_M,
    MAX(o.arr_usd_M) AS max_arr_usd_M
FROM dim_company c
LEFT JOIN fct_article a ON c.company_id = a.company_id
LEFT JOIN fct_arr_observation o ON a.article_id = o.article_id
GROUP BY c.industry
ORDER BY total_articles DESC;
"""

df_result2 = duckdb.query(query2).df()
display(df_result2)

,industry,total_companies,total_articles,avg_arr_usd_M,max_arr_usd_M
0,Data Analytics,7,230,13864.3,83850
1,Cloud Computing,5,176,12722.6,98880
2,AI/ML,3,107,13666.4,62100
3,Cybersecurity,2,74,1075.1,1647
4,SaaS,2,73,35670.1,102600
5,FinTech,2,65,11853.0,37400
6,NaN,5,25,110.3,260


--- 
### **Method 2: Zero-Dependency Built-in SQLite Engine**
If DuckDB is not installed, you can use Python's built-in `sqlite3` library.

In [3]:
import sqlite3

# 1. Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# 2. Register DataFrames as SQL tables
dim_company.to_sql('dim_company', conn, index=False, if_exists='replace')
fct_arr_observation.to_sql('fct_arr_observation', conn, index=False, if_exists='replace')

# 3. Execute SQL Query
sql = """
SELECT 
    c.company_name,
    MAX(o.arr_usd_M) AS peak_arr_M
FROM fct_arr_observation o
JOIN dim_company c ON o.company_id = c.company_id
GROUP BY c.company_name
ORDER BY peak_arr_M DESC
LIMIT 5;
"""

df_top5 = pd.read_sql_query(sql, conn)
display(df_top5)

,company_name,peak_arr_M
0,Amazon Web Services,102600
1,Tesla,98880
2,Microsoft,83850
3,NVIDIA,62100
4,Uber,37400
